# RunPod: which entities of the state does a chunked behaviour-cloning model need?

Run the cells from the top, on an A100 pod (about $1.8/h; the whole notebook takes well under an hour, most of it is reading the `state` column). Once per pod, in a terminal: `git clone https://github.com/quastAI/AAC.git /root/AAC`, then open this notebook from there. The dataset comes from the Hub repo that `runpod_dataset.ipynb` filled (the files must be relabeled, see `relabel_dataset.ipynb`). The helpers are in `notebooks/pod.py`.

This notebook does one thing. `H-JEPA/h_jepa/bc.py` trains an MLP on the **simulator state** (the symbolic observation the expert acted on: no pixels, no encoder, no history) to predict the expert's next `CHUNKS` actions, and then asks which parts of the state the model needs to do it. The result says what a behaviour-cloning term on the flat model's latent would demand of it, and which of that the latent of F1 does not hold. The state is simulator ground truth: this is an analysis, the world models never see it.

| step | what |
|---|---|
| 0 | configuration, Hugging Face login, the PyTorch environment, the dataset from the Hub (the helpers of the flat notebook) |
| 1 | smoke run on 20 episodes: proves the columns and the reports on this pod |
| 2 | one model per chunk length (the final expert's training episodes, `source` 0), its **importance of every entity** on the held-out validation worlds, and the model trained again without the coarse groups and the most important entities |
| 2b | the **temporal-distance** task on the same pipeline: a model reads the state of frame t and of a later frame (1 to 64 steps) and predicts the gap; which entities it needs, to set next to what the actions need |
| 3 | the reports, the figures |
| 4 | the reports to the Hub |

**What is measured.** *Skill* is what the model gains over the marginal action distribution (nats of cross-entropy). The importance of an entity is the share of that skill lost when the entity's columns are replaced by those of another random frame (a block type, a mob type, a ring of tiles around the player, one tile, an inventory item, a vital, facing, daylight, sleeping; all columns of one entity together, so a tile stays one-hot). Permuting measures reliance, not necessity: what another entity duplicates shows little. So the groups and the top entities are also **trained again without** the entity (its columns zero in training and evaluation).

**Temporal distance (2b).** What would a temporal-distance term ask of a latent that a behaviour-cloning term does not? The same measurement with another target: the entities that matter for the distance and not for the actions are the addition. Daylight is left out of it (the light level is a clock and would give the gap away); only the final expert's paths are measured, not the plans of a planner.

**Reading it.** The actions are the executed ones (noop while the player is asleep, as the world models train on them): the sleeping flag then explains the 17% of steps taken asleep, which is no decision of the expert, and its share is large. `ACTIONS = "action_policy"` takes the policy's choice instead. The expert is recurrent, so part of its behaviour is memory, which no state shows: the skill is a lower bound of what the expert's observation allows.

In [ ]:
# --- 0. Configuration ------------------------------------------------------------------------------------------------
import sys
from pathlib import Path

REPO = Path("/root/AAC")                   # the git clone, on the local disk (pod.py has the other paths and why they are where they are)
sys.path.insert(0, str(REPO / "notebooks"))
import pod

HF_REPO_ID = "quastAI/craftax-classic-pqn-rnn-v1"   # the dataset repo of runpod_dataset.ipynb (data/); the reports go to reports/
HF_PRIVATE = True

CHUNKS = (4, 8, 16)                        # actions predicted per frame; a model trains in about a minute on an A100, so all are cheap (to add a length later, run only it: CHUNKS = (16,))
DISTANCE = True                            # also the temporal-distance task (step 2b)
ACTIONS = "action"                         # the executed action (noop while asleep, what the world models train on); "action_policy": the policy's choice
EPOCHS = 8                                 # the model with the lowest validation cross-entropy of these epochs is kept
RETRAIN_TOP = 6                            # entities besides the six coarse groups that are also trained again without
SMOKE_EPISODES = 20                        # episodes per file of the smoke run

In [ ]:
# --- 0. Paths, Hugging Face login, the PyTorch environment (installed again on a new pod) and the dataset ----------------
pod.configure(repo=REPO)
from pod import DATA, H_JEPA, LOGS, PYTHON, launch, progress, running, sh, show, wait

api = pod.login(HF_REPO_ID, HF_PRIVATE)
pod.install(api, HF_REPO_ID)

## 1. Smoke run

One epoch on the first 20 episodes of the training and the validation file, one chunk length: the real command on the real files, so that the columns, the paths and the reports are proven on this pod before the full run. The numbers of a smoke run mean nothing.

In [ ]:
# --- 1. Smoke run on the real files ----------------------------------------------------------------------------------------
SMOKE = DATA / "reports" / "bc_smoke"
sh([PYTHON, "bc.py", "--chunk", 4, "--epochs", 1, "--max-episodes", SMOKE_EPISODES, "--boot", 20, "--repeats", 1,
    "--retrain-top", 1, "--actions", ACTIONS, "--out", SMOKE], cwd=H_JEPA, quiet=True, log="bc_smoke")
assert (SMOKE / "bc_entities_chunk4.md").exists(), "the smoke run wrote no report"
show(SMOKE / "bc_entities_chunk4.md", 20)

## 2. The models and the importance of every entity

`bc.py` reads the `state` and the actions of the final expert's episodes (`source` 0) of the training file, trains one model per chunk length on them and measures the importance on the validation file (held-out worlds, the same expert). Each chunk length writes `bc_entities_chunk<N>.md` (tables per family of entities, the importance by place in the view, the retraining) and `.json` (every number) to `reports/`. The run is detached like the training jobs of the other notebooks: if the pod restarts, run the cell again.

In [ ]:
# --- 2. Start the models, detached ---------------------------------------------------------------------------------------
launch("bc", [PYTHON, "bc.py", "--chunk", *CHUNKS, "--actions", ACTIONS, "--epochs", EPOCHS, "--retrain-top", RETRAIN_TOP,
              "--out", DATA / "reports"], cwd=H_JEPA)

In [ ]:
# --- 2. Wait for it: the last progress line every 30 s -----------------------------------------------------------------------
wait("bc", "[bc]", every=30)
for chunk in CHUNKS:
    assert (DATA / "reports" / f"bc_entities_chunk{chunk}.md").exists(), f"no report for chunk {chunk}: read {LOGS / 'bc.log'}"

## 2b. Temporal distance

The same pipeline with another target: the model reads the state of frame t and of the frame k later (k from 1 to 64 steps, 12 classes, one k per frame of the final expert's episodes) and predicts k. An entity is replaced in both frames. The report is `bc_entities_distance.md` and `.json`. It runs detached under its own name, so it can run next to step 2 on the same GPU (the models are small); `DISTANCE = False` skips it.

In [ ]:
# --- 2b. Temporal distance, detached ------------------------------------------------------------------------------------------
if DISTANCE:
    launch("bc_distance", [PYTHON, "bc.py", "--chunk", "--distance", "--epochs", EPOCHS, "--retrain-top", RETRAIN_TOP,
                           "--out", DATA / "reports"], cwd=H_JEPA)   # `--chunk` without a value: no action task here

In [ ]:
# --- 2b. Wait for it: the last progress line every 30 s -----------------------------------------------------------------------
if DISTANCE:
    wait("bc_distance", "[bc]", every=30)
    assert (DATA / "reports" / "bc_entities_distance.md").exists(), f"no distance report: read {LOGS / 'bc_distance.log'}"

## 3. The reports and the figures

In [ ]:
# --- 3. The reports -----------------------------------------------------------------------------------------------------
for chunk in CHUNKS:
    show(DATA / "reports" / f"bc_entities_chunk{chunk}.md", 400)
if DISTANCE:
    show(DATA / "reports" / "bc_entities_distance.md", 400)

In [ ]:
# --- 3. Figures: the entities by skill lost, and where in the view -----------------------------------------------------------
PLOT = '''
import json, sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

reports, names = Path(sys.argv[1]), sys.argv[2:]   # "chunk4", ..., "distance"
fig, axes = plt.subplots(len(names), 2, figsize=(15, 6 * len(names)), squeeze=False)
for (bars, grid), name in zip(axes, names):
    res = json.loads((reports / f"bc_entities_{name}.json").read_text())
    ent = res["entities"]
    # block and mob types, items, vitals, facing: the tiles, rings and groups cut the same columns again
    names = sorted((n for n in ent if n.split(":")[0] in ("block", "mob", "inventory", "vitals", "state") and not ent[n]["constant"]),
                   key=lambda n: -ent[n]["skill_lost"])[:25][::-1]
    lost = np.array([ent[n]["skill_lost"] for n in names])
    ci = np.array([ent[n]["skill_lost_ci"] for n in names]).T
    bars.barh(names, lost, xerr=[np.clip(lost - ci[0], 0, None), np.clip(ci[1] - lost, 0, None)], color="#4c78a8")
    bars.set_xlabel("skill lost when the entity is replaced by another frame's (95% episode bootstrap)")
    bars.set_title(f"{name}: accuracy per step " + ", ".join(f"{a:.2f}" for a in res["accuracy_steps"]))
    rows, cols = range(-3, 4), range(-4, 5)
    heat = np.array([[ent[f"tile:{r:+d},{c:+d}"]["skill_lost"] for c in cols] for r in rows])
    image = grid.imshow(heat, cmap="viridis")
    for i, r in enumerate(rows):
        for j, c in enumerate(cols):
            grid.text(j, i, f"{heat[i, j]:.2f}", ha="center", va="center", color="w" if heat[i, j] < heat.max() / 2 else "k", fontsize=8)
    grid.set_xticks(range(9), [f"{c:+d}" for c in cols]), grid.set_yticks(range(7), [f"{r:+d}" for r in rows])
    grid.set_title(f"{name}: skill lost by tile (the player at 0,0; up is up)")
    fig.colorbar(image, ax=grid, fraction=0.04)
fig.tight_layout()
fig.savefig(reports / "bc_entities.png", dpi=150)
print("saved", reports / "bc_entities.png")
'''
sh([PYTHON, "-c", PLOT, DATA / "reports", *[f"chunk{c}" for c in CHUNKS], *(["distance"] if DISTANCE else [])])
from IPython.display import Image, display
display(Image(filename=str(DATA / "reports" / "bc_entities.png")))

## 4. Reports to the Hub

Only the reports of this notebook go to `reports/` of the Hub repo (the figure, the markdown and the JSON of each chunk length); the runs are not touched.

In [ ]:
# --- 4. The reports to the Hub --------------------------------------------------------------------------------------------
api.upload_folder(repo_id=HF_REPO_ID, repo_type="dataset", folder_path=str(DATA / "reports"), path_in_repo="reports",
                  allow_patterns=["bc_entities_*", "bc_entities.png"])
print(f"done: https://huggingface.co/datasets/{HF_REPO_ID}/tree/main/reports")